# フォントファイルの構造を理解する

TTF/OTF フォントファイルは **テーブルの集合体** です。  
このノートブックでは `fontTools` を使って各テーブルの中身を段階的に確認します。

In [1]:
from fontTools.ttLib import TTFont

FONT_PATH = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
font = TTFont(FONT_PATH, lazy=True)
print("読み込み完了:", FONT_PATH)

読み込み完了: /usr/share/fonts/truetype/dejavu/DejaVuSans.ttf


## 1. フォントに含まれるテーブル一覧

`TTFont` は辞書ライクなオブジェクトです。  
キーがテーブルタグ（4文字）、値が各テーブルの解析済みオブジェクトになっています。

In [ ]:
tables = font.keys()
print(f"テーブル数: {len(tables)}")
print(sorted(tables))

テーブル数: 21
['FFTM', 'GDEF', 'GPOS', 'GSUB', 'GlyphOrder', 'MATH', 'OS/2', 'cmap', 'cvt ', 'fpgm', 'gasp', 'glyf', 'head', 'hhea', 'hmtx', 'kern', 'loca', 'maxp', 'name', 'post', 'prep']


主なテーブルの役割：

| タグ | 内容 |
|------|------|
| `name` | フォント名・バージョン・ライセンスなどの文字列情報 |
| `OS/2` | 太さ・幅・対応 Unicode レンジなど |
| `cmap` | Unicode コードポイント → グリフ名 のマッピング |
| `post` | 等幅フラグ・イタリック角度・PostScript名 |
| `head` | フォントのバージョン・作成日・units per em |
| `glyf` | グリフの輪郭データ（ベジェ曲線）|
| `hhea` | 水平方向のメトリクス（ascent / descent）|

## 2. `name` テーブル — フォントのメタデータ

`name` テーブルは「nameID → 文字列」のリストです。  
同じ nameID が複数の言語・プラットフォーム向けに格納されていることがあります。

In [3]:
NAME_IDS = {
    0: "copyright",
    1: "family",
    2: "subfamily",
    3: "unique_id",
    4: "full_name",
    5: "version",
    6: "postscript_name",
    7: "trademark",
    8: "manufacturer",
    9: "designer",
    13: "license",
    14: "license_url",
}

for nid, label in NAME_IDS.items():
    # Windows / BMP / 英語 を優先、なければ Mac fallback
    record = font["name"].getName(nid, 3, 1, 0x0409)
    if record is None:
        record = font["name"].getName(nid, 1, 0, 0)
    value = record.toUnicode() if record else None
    # ライセンスは長いので先頭50文字だけ
    if value and len(value) > 50:
        value = value[:50] + "..."
    print(f"  [{nid:2d}] {label:20s}: {value}")

  [ 0] copyright           : Copyright (c) 2003 by Bitstream, Inc. All Rights R...
  [ 1] family              : DejaVu Sans
  [ 2] subfamily           : Book
  [ 3] unique_id           : DejaVu Sans
  [ 4] full_name           : DejaVu Sans
  [ 5] version             : Version 2.37
  [ 6] postscript_name     : DejaVuSans
  [ 7] trademark           : None
  [ 8] manufacturer        : DejaVu fonts team
  [ 9] designer            : None
  [13] license             : Fonts are (c) Bitstream (see below). DejaVu change...
  [14] license_url         : http://dejavu.sourceforge.net/wiki/index.php/Licen...


## 3. `OS/2` テーブル — 太さ・幅・スタイル

フォントの視覚的な属性を定義します。

In [4]:
os2 = font["OS/2"]

WEIGHT_NAMES = {
    100: "Thin", 200: "ExtraLight", 300: "Light", 400: "Regular",
    500: "Medium", 600: "SemiBold", 700: "Bold", 800: "ExtraBold", 900: "Black"
}
WIDTH_NAMES = {
    1: "UltraCondensed", 2: "ExtraCondensed", 3: "Condensed", 4: "SemiCondensed",
    5: "Normal", 6: "SemiExpanded", 7: "Expanded", 8: "ExtraExpanded", 9: "UltraExpanded"
}

print(f"weight_class : {os2.usWeightClass} ({WEIGHT_NAMES.get(os2.usWeightClass, '?')})")
print(f"width_class  : {os2.usWidthClass}  ({WIDTH_NAMES.get(os2.usWidthClass, '?')})")

weight_class : 400 (Regular)
width_class  : 5  (Normal)


## 4. `post` テーブル — 等幅・イタリック角度

In [5]:
post = font["post"]

print(f"is_monospaced : {bool(post.isFixedPitch)}")
print(f"italic_angle  : {post.italicAngle}")

is_monospaced : False
italic_angle  : 0.0


## 5. `cmap` テーブル — コードポイント → グリフのマッピング

`cmap` はフォントが**どの文字をサポートしているか**を定義します。  
キーが Unicode コードポイント（整数）、値がグリフ名（文字列）です。

In [6]:
cmap = font.getBestCmap()   # 最適な cmap サブテーブルを自動選択

print(f"サポートするコードポイント数: {len(cmap)}")
print()

# 先頭10件を確認
print("コードポイント → グリフ名 (先頭10件)")
for cp, glyph_name in list(cmap.items())[:10]:
    char = chr(cp)
    print(f"  U+{cp:04X}  '{char}'  → {glyph_name}")

サポートするコードポイント数: 5918

コードポイント → グリフ名 (先頭10件)
  U+0020  ' '  → space
  U+0021  '!'  → exclam
  U+0022  '"'  → quotedbl
  U+0023  '#'  → numbersign
  U+0024  '$'  → dollar
  U+0025  '%'  → percent
  U+0026  '&'  → ampersand
  U+0027  '''  → quotesingle
  U+0028  '('  → parenleft
  U+0029  ')'  → parenright


In [7]:
# 特定の文字がグリフを持つか確認
test_chars = ["A", "a", "あ", "漢", "α", "→"]

print("グリフ存在チェック")
for c in test_chars:
    has = ord(c) in cmap
    mark = "✓" if has else "✗"
    print(f"  {mark} U+{ord(c):04X} '{c}'")

グリフ存在チェック
  ✓ U+0041 'A'
  ✓ U+0061 'a'
  ✗ U+3042 'あ'
  ✗ U+6F22 '漢'
  ✓ U+03B1 'α'
  ✓ U+2192 '→'


## 6. `head` テーブル — 基本情報

In [8]:
head = font["head"]

print(f"units_per_em  : {head.unitsPerEm}")
print(f"font_revision : {head.fontRevision}")

units_per_em  : 2048
font_revision : 2.3699951171875


`units_per_em` はフォントの座標系の精度を表します（通常 1000 または 2048）。  
グリフの輪郭データはこの単位系で記述されており、レンダリング時に実ピクセルへスケールされます。

## 7. 複数フォントの比較

Regular と Bold で `weight_class` がどう変わるか確認します。

In [9]:
font_paths = {
    "DejaVuSans": "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "DejaVuSans-Bold": "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
}

print(f"{'name':20s}  {'weight':>6}  {'monospaced':>10}  {'glyphs':>6}")
print("-" * 50)

for name, path in font_paths.items():
    f = TTFont(path, lazy=True)
    w = f["OS/2"].usWeightClass
    mono = bool(f["post"].isFixedPitch)
    n = len(f.getBestCmap() or {})
    print(f"{name:20s}  {w:>6}  {str(mono):>10}  {n:>6}")

name                  weight  monospaced  glyphs
--------------------------------------------------
DejaVuSans               400       False    5918
DejaVuSans-Bold          700       False    5898


## まとめ

| テーブル | 取得できる情報 | 主な用途 |
|---------|-------------|--------|
| `name`  | ファミリー名・バージョン・ライセンス | メタデータ収集 |
| `OS/2`  | 太さ・幅 | フォント分類 |
| `post`  | 等幅・イタリック角度 | フォント分類 |
| `cmap`  | サポートするコードポイント一覧 | グリフ存在チェック |
| `head`  | units per em | レンダリング計算 |

`fontreader.py` はこれらを `FontMetadata` にまとめて返します。